# Fable5 Pipeline — Colab Demo

This notebook runs the Fable5 multi-step reasoning pipeline on a free Colab T4 GPU.

**Architecture:** Planner → Drafter(s) → Verifier → Fuser

Uses Mistral-7B-Instruct with 4-bit quantization (bitsandbytes).

In [ ]:
# Step 1: Install dependencies
!pip install -q transformers accelerate bitsandbytes peft sentence-transformers faiss-cpu tqdm pydantic

In [ ]:
# Step 2: Clone the repo
!git clone https://github.com/yourname/fable5-pipeline.git 2>/dev/null || true
%cd fable5-pipeline
!ls

In [ ]:
# Step 3: Import and set up
import sys
sys.path.insert(0, '.')

from src.inference_backends import HfBnbRunner
from src.orchestrator import Orchestrator

print('Loading model (this may take 2-3 minutes)...')
backend = HfBnbRunner(model_id='mistralai/Mistral-7B-Instruct-v0.3')
orch = Orchestrator(backend, k_candidates=2, log_dir='logs')
print('Model loaded!')

In [ ]:
# Step 4: Run the pipeline!
goal = "Explain the concept of supply and demand with real-world examples from the housing market."

print(f'Goal: {goal}')
print('=' * 60)

trace = orch.run(goal)

print('\n' + '=' * 60)
print('PLAN:')
for i, step in enumerate(trace['plan'].get('steps', []), 1):
    print(f'  {i}. {step}')
print(f'  DoD: {trace["plan"].get("dod", "")}')

print('\n' + '=' * 60)
print('FINAL ANSWER:')
print(trace['final_answer'])

print('\n' + '=' * 60)
print('METRICS:')
for k, v in trace['metrics'].items():
    print(f'  {k}: {v}')

In [ ]:
# Step 5: Try your own prompt!
# Uncomment and modify the line below:
# trace = orch.run("Your question here")

## Next Steps

- Run the evaluation harness: `!python -m tests.eval_harness --prompts tests/sample_prompts.jsonl --backend hf_bnb --model-id mistralai/Mistral-7B-Instruct-v0.3`
- Fine-tune with LoRA: `!python src/train_lora.py --dataset data/instructions.jsonl --model-id mistralai/Mistral-7B-Instruct-v0.3`
- Save logs to Google Drive before session ends!